# DriftGuard — Git History Extraction

This notebook extracts real configuration-file changes from complete Git histories.

For every relevant file modification, it records:

- Repository and dataset split
- Commit and parent commit
- Commit timestamp and message
- Old and new file paths
- Configuration type
- File contents before and after the change
- Added and deleted line counts
- Content hashes for deduplication

The validation and test repositories are extracted for later evaluation, but they will not be used for model fitting or threshold selection.

In [1]:
import os
import sys
import json
import gzip
import hashlib
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import pandas as pd
from tqdm.auto import tqdm
from pydriller import Repository

print("=" * 72)
print("DRIFTGUARD — GIT HISTORY EXTRACTION")
print("=" * 72)

current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

REPOSITORIES_DIR = PROJECT_ROOT / "repositories"
CONFIGS_DIR = PROJECT_ROOT / "configs"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
LOGS_DIR = PROJECT_ROOT / "logs"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Python version      :", sys.version.split()[0])
print("Python executable   :", sys.executable)
print("Conda environment   :", os.environ.get("CONDA_DEFAULT_ENV", "Not detected"))
print("Platform            :", platform.platform())
print("Project root        :", PROJECT_ROOT)
print("Repositories folder :", REPOSITORIES_DIR)
print("Raw data folder     :", RAW_DATA_DIR)

DRIFTGUARD — GIT HISTORY EXTRACTION
Python version      : 3.11.15
Python executable   : C:\Users\Lenovo\anaconda3\envs\driftguard\python.exe
Conda environment   : driftguard
Platform            : Windows-10-10.0.26200-SP0
Project root        : C:\Users\Lenovo\Desktop\DriftGuard
Repositories folder : C:\Users\Lenovo\Desktop\DriftGuard\repositories
Raw data folder     : C:\Users\Lenovo\Desktop\DriftGuard\data\raw


In [2]:
%pip install --upgrade ipywidgets jupyterlab_widgets widgetsnbextension

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


In [2]:
project_settings_path = CONFIGS_DIR / "project_settings.json"
repository_config_path = CONFIGS_DIR / "repositories.json"

if not project_settings_path.exists():
    raise FileNotFoundError(
        f"Missing project settings file:\n{project_settings_path}\n"
        "Run Notebook 01 first."
    )

if not repository_config_path.exists():
    raise FileNotFoundError(
        f"Missing repository configuration file:\n{repository_config_path}\n"
        "Run Notebook 01 first."
    )

with project_settings_path.open("r", encoding="utf-8") as file:
    PROJECT_SETTINGS = json.load(file)

with repository_config_path.open("r", encoding="utf-8") as file:
    repository_configuration = json.load(file)

REPOSITORIES = repository_configuration["repositories"]

CONFIG_EXTENSIONS = {
    extension.lower()
    for extension in PROJECT_SETTINGS["config_extensions"]
}

CONFIG_FILENAMES = {
    filename.lower()
    for filename in PROJECT_SETTINGS["config_filenames"]
}

MAX_FILE_SIZE_BYTES = int(
    PROJECT_SETTINGS.get("max_file_size_bytes", 1_000_000)
)

print("Repositories loaded :", len(REPOSITORIES))
print("Configuration types :", len(CONFIG_EXTENSIONS))
print("Maximum file size   :", f"{MAX_FILE_SIZE_BYTES:,} bytes")

display(
    pd.DataFrame(REPOSITORIES)[
        ["name", "split", "url", "primary_config_types"]
    ]
)

Repositories loaded : 6
Configuration types : 11
Maximum file size   : 1,000,000 bytes


,name,split,url,primary_config_types
0,microservices_demo,train,https://github.com/GoogleCloudPlatform/microse...,"[kubernetes, helm, kustomize, terraform]"
1,kube_prometheus,train,https://github.com/prometheus-operator/kube-pr...,"[kubernetes, jsonnet, yaml]"
2,terraform_aws_vpc,train,https://github.com/terraform-aws-modules/terra...,[terraform]
3,kubernetes_examples,validation,https://github.com/kubernetes/examples.git,"[kubernetes, yaml]"
4,ansible_examples,test,https://github.com/ansible/ansible-examples.git,"[ansible, nginx, yaml]"
5,terraform_eks_blueprints,test,https://github.com/aws-ia/terraform-aws-eks-bl...,"[terraform, kubernetes]"


In [3]:
def run_command(command, cwd=None, check=True):
    """Run a terminal command and return the completed process."""

    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(map(str, command))}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    return result


git_version = run_command(["git", "--version"]).stdout.strip()

try:
    import pydriller
    pydriller_version = getattr(pydriller, "__version__", "unknown")
except Exception:
    pydriller_version = "unknown"

print("Git version      :", git_version)
print("PyDriller version:", pydriller_version)
print("Environment check: PASSED")

Git version      : git version 2.51.2.windows.1
PyDriller version: 2.10
Environment check: PASSED


In [4]:
repository_status_records = []

for repository in REPOSITORIES:
    repository_path = REPOSITORIES_DIR / repository["name"]
    git_directory = repository_path / ".git"

    available = repository_path.exists() and git_directory.exists()

    if available:
        head_commit = run_command(
            [
                "git",
                "-C",
                str(repository_path),
                "rev-parse",
                "HEAD",
            ],
            check=False,
        ).stdout.strip()

        no_merge_commit_count_result = run_command(
            [
                "git",
                "-C",
                str(repository_path),
                "rev-list",
                "--count",
                "--no-merges",
                "HEAD",
            ],
            check=False,
        )

        no_merge_commit_count = (
            int(no_merge_commit_count_result.stdout.strip())
            if no_merge_commit_count_result.returncode == 0
            and no_merge_commit_count_result.stdout.strip().isdigit()
            else None
        )
    else:
        head_commit = None
        no_merge_commit_count = None

    repository_status_records.append(
        {
            "repository": repository["name"],
            "split": repository["split"],
            "available": available,
            "head_commit": head_commit,
            "non_merge_commits": no_merge_commit_count,
            "local_path": str(repository_path),
        }
    )

repository_status = pd.DataFrame(repository_status_records)

display(repository_status)

if not repository_status["available"].all():
    unavailable = repository_status.loc[
        ~repository_status["available"],
        "repository",
    ].tolist()

    raise RuntimeError(
        "The following repositories are unavailable: "
        + ", ".join(unavailable)
    )

print("\nAll repositories are available.")

,repository,split,available,head_commit,non_merge_commits,local_path
0,microservices_demo,train,True,9a4616e77f0f9cbcbecaf27d711c38890dda1404,2622,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
1,kube_prometheus,train,True,f912700123916f3ecc80bbb6dd61868cabe9a02f,2336,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
2,terraform_aws_vpc,train,True,3ffbd46fb1c7733e1b34d8666893280454e27436,497,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
3,kubernetes_examples,validation,True,d6b8cd27eacb51e651a1aa6f7c190a28713eff6e,1411,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
4,ansible_examples,test,True,b50586543c6c4be907fdc88f9f78a2b35d2a895f,313,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...
5,terraform_eks_blueprints,test,True,eec18146cc2d883e093c548de0a92a5d510205d1,1377,C:\Users\Lenovo\Desktop\DriftGuard\repositorie...



All repositories are available.


In [5]:
SPECIAL_CONFIG_NAMES = {
    "chart.yaml",
    "chart.yml",
    "values.yaml",
    "values.yml",
    "kustomization.yaml",
    "kustomization.yml",
    "dockerfile",
    "containerfile",
    "ansible.cfg",
    "nginx.conf",
    "httpd.conf",
    "prometheus.yml",
    "prometheus.yaml",
    "grafana.ini",
    "terraform.tfvars",
}

EXCLUDED_PATH_PARTS = {
    ".git",
    "node_modules",
    "vendor",
    ".terraform",
    "__pycache__",
    "dist",
    "build",
    "coverage",
}

CONFIG_DIRECTORY_HINTS = {
    "kubernetes",
    "k8s",
    "manifests",
    "manifest",
    "helm",
    "charts",
    "chart",
    "terraform",
    "ansible",
    "playbooks",
    "roles",
    "nginx",
    "config",
    "configs",
    "deployment",
    "deployments",
    "infrastructure",
    "infra",
}


def normalize_git_path(file_path):
    """Normalize a repository-relative path."""

    if file_path is None:
        return None

    return str(file_path).replace("\\", "/").strip()


def is_excluded_path(file_path):
    """Return True for generated or dependency directories."""

    normalized_path = normalize_git_path(file_path)

    if not normalized_path:
        return False

    path_parts = {
        part.lower()
        for part in Path(normalized_path).parts
    }

    return bool(path_parts.intersection(EXCLUDED_PATH_PARTS))


def is_configuration_file(file_path):
    """Determine whether a path represents a configuration file."""

    normalized_path = normalize_git_path(file_path)

    if not normalized_path:
        return False

    if is_excluded_path(normalized_path):
        return False

    path = Path(normalized_path)
    filename = path.name.lower()
    suffix = path.suffix.lower()

    if filename in CONFIG_FILENAMES:
        return True

    if filename in SPECIAL_CONFIG_NAMES:
        return True

    if suffix in CONFIG_EXTENSIONS:
        return True

    return False


def infer_configuration_type(file_path):
    """Infer the configuration technology from a file path."""

    normalized_path = normalize_git_path(file_path)

    if not normalized_path:
        return "unknown"

    lower_path = normalized_path.lower()
    path = Path(lower_path)
    filename = path.name
    suffix = path.suffix

    if suffix in {".tf", ".tfvars", ".hcl"}:
        return "terraform"

    if filename in {
        "chart.yaml",
        "chart.yml",
        "values.yaml",
        "values.yml",
    } or "/charts/" in lower_path or "/helm/" in lower_path:
        return "helm"

    if filename in {
        "kustomization.yaml",
        "kustomization.yml",
    } or "kustomize" in lower_path:
        return "kustomize"

    if filename in {"nginx.conf"} or "nginx" in lower_path:
        return "nginx"

    if filename in {"dockerfile", "containerfile"}:
        return "docker"

    if filename in {
        "docker-compose.yml",
        "docker-compose.yaml",
    }:
        return "docker_compose"

    if filename == "ansible.cfg":
        return "ansible"

    if any(
        token in lower_path
        for token in ["/playbooks/", "/roles/", "ansible"]
    ):
        return "ansible"

    if suffix in {".yaml", ".yml"}:
        if any(
            token in lower_path
            for token in [
                "kubernetes",
                "/k8s/",
                "/manifests/",
                "/deployments/",
            ]
        ):
            return "kubernetes"

        return "yaml"

    if suffix == ".json":
        return "json"

    if suffix == ".toml":
        return "toml"

    if suffix in {".ini", ".cfg"}:
        return "ini"

    if suffix in {".conf", ".properties"}:
        return "configuration"

    return "other"

In [6]:
test_paths = [
    "kubernetes/deployment.yaml",
    "terraform/main.tf",
    "charts/frontend/values.yaml",
    "roles/nginx/tasks/main.yml",
    "nginx/sites-enabled/default.conf",
    "Dockerfile",
    "src/application.py",
    "node_modules/package/config.json",
    "README.md",
]

detection_test_records = []

for test_path in test_paths:
    detection_test_records.append(
        {
            "path": test_path,
            "is_configuration": is_configuration_file(test_path),
            "configuration_type": (
                infer_configuration_type(test_path)
                if is_configuration_file(test_path)
                else None
            ),
        }
    )

detection_test_df = pd.DataFrame(detection_test_records)

display(detection_test_df)

expected_results = {
    "kubernetes/deployment.yaml": True,
    "terraform/main.tf": True,
    "charts/frontend/values.yaml": True,
    "roles/nginx/tasks/main.yml": True,
    "nginx/sites-enabled/default.conf": True,
    "Dockerfile": True,
    "src/application.py": False,
    "node_modules/package/config.json": False,
    "README.md": False,
}

for path, expected in expected_results.items():
    actual = bool(
        detection_test_df.loc[
            detection_test_df["path"] == path,
            "is_configuration",
        ].iloc[0]
    )

    assert actual == expected, (
        f"Configuration detection failed for {path}: "
        f"expected {expected}, received {actual}"
    )

print("Configuration detection tests: PASSED")

,path,is_configuration,configuration_type
0,kubernetes/deployment.yaml,True,kubernetes
1,terraform/main.tf,True,terraform
2,charts/frontend/values.yaml,True,helm
3,roles/nginx/tasks/main.yml,True,nginx
4,nginx/sites-enabled/default.conf,True,nginx
5,Dockerfile,True,docker
6,src/application.py,False,NaN
7,node_modules/package/config.json,False,NaN
8,README.md,False,NaN


Configuration detection tests: PASSED


In [7]:
def convert_to_text(value):
    """Safely convert source content to text."""

    if value is None:
        return None

    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")

    return str(value)


def encoded_size(value):
    """Return UTF-8 encoded size in bytes."""

    if value is None:
        return 0

    return len(value.encode("utf-8", errors="replace"))


def count_text_lines(value):
    """Count lines in a text value."""

    if value is None or value == "":
        return 0

    return len(value.splitlines())


def sha256_text(value):
    """Return the SHA-256 hash of a text value."""

    if value is None:
        return None

    return hashlib.sha256(
        value.encode("utf-8", errors="replace")
    ).hexdigest()


def safe_isoformat(value):
    """Convert datetime-like values to ISO 8601."""

    if value is None:
        return None

    if hasattr(value, "isoformat"):
        return value.isoformat()

    return str(value)


def create_record_id(
    repository_name,
    commit_hash,
    old_path,
    new_path,
    modified_file_index,
):
    """Create a stable identifier for a file-change record."""

    identifier_text = "|".join(
        [
            str(repository_name),
            str(commit_hash),
            str(old_path),
            str(new_path),
            str(modified_file_index),
        ]
    )

    return hashlib.sha256(
        identifier_text.encode("utf-8")
    ).hexdigest()

In [8]:
EXTRACTION_SETTINGS = {
    # Set to True only when you intentionally want to overwrite outputs.
    "force_reextract": False,

    # None means scan the complete available non-merge history.
    "max_commits_per_repository": None,

    # Merge commits are excluded to reduce duplicate changes and leakage.
    "exclude_merge_commits": True,

    # Large files are skipped to control memory and dataset size.
    "maximum_file_size_bytes": MAX_FILE_SIZE_BYTES,

    # Source before and after are retained for structural diff generation.
    "retain_source_contents": True,
}

extraction_settings_path = (
    CONFIGS_DIR / "git_extraction_settings.json"
)

with extraction_settings_path.open("w", encoding="utf-8") as file:
    json.dump(EXTRACTION_SETTINGS, file, indent=2)

print(json.dumps(EXTRACTION_SETTINGS, indent=2))
print("\nSaved to:", extraction_settings_path)

{
  "force_reextract": false,
  "max_commits_per_repository": null,
  "exclude_merge_commits": true,
  "maximum_file_size_bytes": 1000000,
  "retain_source_contents": true
}

Saved to: C:\Users\Lenovo\Desktop\DriftGuard\configs\git_extraction_settings.json


In [9]:
def get_non_merge_commit_count(repository_path):
    """Return the number of non-merge commits reachable from HEAD."""

    result = run_command(
        [
            "git",
            "-C",
            str(repository_path),
            "rev-list",
            "--count",
            "--no-merges",
            "HEAD",
        ],
        check=False,
    )

    if (
        result.returncode == 0
        and result.stdout.strip().isdigit()
    ):
        return int(result.stdout.strip())

    return None


def extract_repository_history(
    repository_config,
    force_reextract=False,
    max_commits=None,
):
    """
    Extract configuration-file modifications from one Git repository.

    A compressed JSONL file is written per repository so that extraction
    can restart without losing completed repositories.
    """

    repository_name = repository_config["name"]
    repository_split = repository_config["split"]
    repository_url = repository_config["url"]
    repository_path = REPOSITORIES_DIR / repository_name

    output_path = (
        RAW_DATA_DIR
        / f"{repository_name}_configuration_changes.jsonl.gz"
    )

    temporary_output_path = output_path.with_suffix(
        output_path.suffix + ".tmp"
    )

    summary_path = (
        RAW_DATA_DIR
        / f"{repository_name}_extraction_summary.json"
    )

    if (
        output_path.exists()
        and summary_path.exists()
        and not force_reextract
    ):
        print(
            f"{repository_name}: existing extraction found; "
            "skipping."
        )

        with summary_path.open("r", encoding="utf-8") as file:
            return json.load(file)

    if not (repository_path / ".git").exists():
        raise FileNotFoundError(
            f"Not a valid Git repository: {repository_path}"
        )

    total_non_merge_commits = get_non_merge_commit_count(
        repository_path
    )

    traversal = Repository(
        path_to_repo=str(repository_path),
        only_no_merge=True,
        order="reverse",
    ).traverse_commits()

    skip_reasons = Counter()
    configuration_type_counts = Counter()
    change_type_counts = Counter()

    traversed_commits = 0
    commits_with_configuration_changes = 0
    extracted_records = 0
    extraction_errors = 0

    earliest_timestamp = None
    latest_timestamp = None

    if temporary_output_path.exists():
        temporary_output_path.unlink()

    progress_total = (
        min(total_non_merge_commits, max_commits)
        if total_non_merge_commits is not None
        and max_commits is not None
        else total_non_merge_commits
    )

    with gzip.open(
        temporary_output_path,
        mode="wt",
        encoding="utf-8",
    ) as output_file:

        progress_bar = tqdm(
            traversal,
            total=progress_total,
            desc=repository_name,
            unit="commit",
        )

        for commit_index, commit in enumerate(progress_bar):
            if max_commits is not None and commit_index >= max_commits:
                break

            traversed_commits += 1
            commit_record_count = 0

            commit_hash = commit.hash
            parent_hash = (
                commit.parents[0]
                if getattr(commit, "parents", None)
                else None
            )

            commit_timestamp = safe_isoformat(
                getattr(commit, "author_date", None)
            )

            if commit_timestamp:
                if (
                    earliest_timestamp is None
                    or commit_timestamp < earliest_timestamp
                ):
                    earliest_timestamp = commit_timestamp

                if (
                    latest_timestamp is None
                    or commit_timestamp > latest_timestamp
                ):
                    latest_timestamp = commit_timestamp

            for modified_file_index, modified_file in enumerate(
                commit.modified_files
            ):
                try:
                    old_path = normalize_git_path(
                        modified_file.old_path
                    )

                    new_path = normalize_git_path(
                        modified_file.new_path
                    )

                    effective_path = new_path or old_path

                    if not effective_path:
                        skip_reasons["missing_path"] += 1
                        continue

                    if not is_configuration_file(effective_path):
                        skip_reasons["not_configuration_file"] += 1
                        continue

                    source_before = convert_to_text(
                        modified_file.source_code_before
                    )

                    source_after = convert_to_text(
                        modified_file.source_code
                    )

                    if source_before is None and source_after is None:
                        skip_reasons[
                            "missing_or_binary_source"
                        ] += 1
                        continue

                    old_size_bytes = encoded_size(source_before)
                    new_size_bytes = encoded_size(source_after)

                    if (
                        old_size_bytes > MAX_FILE_SIZE_BYTES
                        or new_size_bytes > MAX_FILE_SIZE_BYTES
                    ):
                        skip_reasons["file_too_large"] += 1
                        continue

                    raw_change_type = getattr(
                        modified_file,
                        "change_type",
                        None,
                    )

                    change_type = getattr(
                        raw_change_type,
                        "name",
                        str(raw_change_type),
                    )

                    configuration_type = infer_configuration_type(
                        effective_path
                    )

                    record = {
                        "record_id": create_record_id(
                            repository_name=repository_name,
                            commit_hash=commit_hash,
                            old_path=old_path,
                            new_path=new_path,
                            modified_file_index=modified_file_index,
                        ),
                        "repository": repository_name,
                        "repository_url": repository_url,
                        "dataset_split": repository_split,
                        "repository_commit_index": commit_index,
                        "commit_hash": commit_hash,
                        "parent_hash": parent_hash,
                        "commit_author_date": commit_timestamp,
                        "commit_committer_date": safe_isoformat(
                            getattr(
                                commit,
                                "committer_date",
                                None,
                            )
                        ),
                        "commit_message": (
                            getattr(commit, "msg", None) or ""
                        ).strip(),
                        "author_name": getattr(
                            getattr(commit, "author", None),
                            "name",
                            None,
                        ),
                        "in_main_branch": bool(
                            getattr(
                                commit,
                                "in_main_branch",
                                True,
                            )
                        ),
                        "is_merge_commit": bool(
                            getattr(commit, "merge", False)
                        ),
                        "modified_file_index": modified_file_index,
                        "filename": modified_file.filename,
                        "old_path": old_path,
                        "new_path": new_path,
                        "effective_path": effective_path,
                        "change_type": change_type,
                        "configuration_type": configuration_type,
                        "added_lines": int(
                            getattr(
                                modified_file,
                                "added_lines",
                                0,
                            )
                            or 0
                        ),
                        "deleted_lines": int(
                            getattr(
                                modified_file,
                                "deleted_lines",
                                0,
                            )
                            or 0
                        ),
                        "old_size_bytes": old_size_bytes,
                        "new_size_bytes": new_size_bytes,
                        "old_line_count": count_text_lines(
                            source_before
                        ),
                        "new_line_count": count_text_lines(
                            source_after
                        ),
                        "source_before_sha256": sha256_text(
                            source_before
                        ),
                        "source_after_sha256": sha256_text(
                            source_after
                        ),
                        "source_before": source_before,
                        "source_after": source_after,
                    }

                    output_file.write(
                        json.dumps(
                            record,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )

                    extracted_records += 1
                    commit_record_count += 1
                    configuration_type_counts[
                        configuration_type
                    ] += 1
                    change_type_counts[change_type] += 1

                except Exception as error:
                    extraction_errors += 1
                    skip_reasons[
                        f"processing_error:{type(error).__name__}"
                    ] += 1

            if commit_record_count > 0:
                commits_with_configuration_changes += 1

            progress_bar.set_postfix(
                records=extracted_records,
                config_commits=commits_with_configuration_changes,
            )

    os.replace(temporary_output_path, output_path)

    summary = {
        "repository": repository_name,
        "repository_url": repository_url,
        "dataset_split": repository_split,
        "repository_path": str(repository_path),
        "output_path": str(output_path),
        "total_non_merge_commits_available": total_non_merge_commits,
        "traversed_commits": traversed_commits,
        "commits_with_configuration_changes":
            commits_with_configuration_changes,
        "extracted_configuration_change_records":
            extracted_records,
        "extraction_errors": extraction_errors,
        "earliest_timestamp": earliest_timestamp,
        "latest_timestamp": latest_timestamp,
        "configuration_type_counts":
            dict(configuration_type_counts),
        "change_type_counts":
            dict(change_type_counts),
        "skip_reasons":
            dict(skip_reasons),
        "head_commit": run_command(
            [
                "git",
                "-C",
                str(repository_path),
                "rev-parse",
                "HEAD",
            ]
        ).stdout.strip(),
        "extracted_at_utc":
            datetime.now(timezone.utc).isoformat(),
    }

    with summary_path.open("w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)

    return summary

In [10]:
extraction_summaries = []

for repository_config in REPOSITORIES:
    print("\n" + "=" * 72)
    print(
        f"Extracting: {repository_config['name']} "
        f"({repository_config['split']})"
    )
    print("=" * 72)

    summary = extract_repository_history(
        repository_config=repository_config,
        force_reextract=EXTRACTION_SETTINGS[
            "force_reextract"
        ],
        max_commits=EXTRACTION_SETTINGS[
            "max_commits_per_repository"
        ],
    )

    extraction_summaries.append(summary)

print("\nAll repository extraction stages completed.")


Extracting: microservices_demo (train)


microservices_demo:   0%|          | 0/2622 [00:00<?, ?commit/s]


Extracting: kube_prometheus (train)


kube_prometheus:   0%|          | 0/2336 [00:00<?, ?commit/s]


Extracting: terraform_aws_vpc (train)


...


Extracting: ansible_examples (test)


ansible_examples:   0%|          | 0/313 [00:00<?, ?commit/s]


Extracting: terraform_eks_blueprints (test)


terraform_eks_blueprints:   0%|          | 0/1377 [00:00<?, ?commit/s]


All repository extraction stages completed.


In [11]:
summary_records = []

for summary in extraction_summaries:
    summary_records.append(
        {
            "repository": summary["repository"],
            "split": summary["dataset_split"],
            "commits_available":
                summary["total_non_merge_commits_available"],
            "commits_scanned":
                summary["traversed_commits"],
            "config_commits":
                summary["commits_with_configuration_changes"],
            "change_records":
                summary[
                    "extracted_configuration_change_records"
                ],
            "errors": summary["extraction_errors"],
            "earliest_timestamp":
                summary["earliest_timestamp"],
            "latest_timestamp":
                summary["latest_timestamp"],
        }
    )

extraction_summary_df = pd.DataFrame(summary_records)

display(extraction_summary_df)

print(
    "\nTotal extracted configuration-file changes:",
    f"{extraction_summary_df['change_records'].sum():,}",
)

,repository,split,commits_available,commits_scanned,config_commits,change_records,errors,earliest_timestamp,latest_timestamp
0,microservices_demo,train,2622,2622,1205,3477,0,2018-06-13T14:22:48-07:00,2026-07-13T10:40:02-04:00
1,kube_prometheus,train,2336,2336,1297,7253,0,2016-10-18T11:03:12+02:00,2026-07-17T22:52:36+00:00
2,terraform_aws_vpc,train,497,497,287,979,0,2017-08-28T18:00:21+02:00,2026-04-02T22:21:35+02:00
3,kubernetes_examples,validation,1411,1411,624,1946,0,2014-06-06T16:40:48-07:00,2026-02-28T15:29:04+05:30
4,ansible_examples,test,313,313,230,932,0,2013-03-12T13:05:13+05:30,2020-07-19T16:39:16-07:00
5,terraform_eks_blueprints,test,1377,1377,1142,7700,0,2021-04-06T17:00:00-07:00,2026-07-07T18:04:37+02:00



Total extracted configuration-file changes: 22,287


In [12]:
SOURCE_CONTENT_FIELDS = {
    "source_before",
    "source_after",
}

metadata_records = []

for repository in REPOSITORIES:
    repository_name = repository["name"]

    data_path = (
        RAW_DATA_DIR
        / f"{repository_name}_configuration_changes.jsonl.gz"
    )

    if not data_path.exists():
        print("Missing extraction file:", data_path)
        continue

    with gzip.open(
        data_path,
        mode="rt",
        encoding="utf-8",
    ) as input_file:

        for line in input_file:
            record = json.loads(line)

            metadata_record = {
                key: value
                for key, value in record.items()
                if key not in SOURCE_CONTENT_FIELDS
            }

            metadata_records.append(metadata_record)

configuration_change_manifest = pd.DataFrame(metadata_records)

print(
    "Global manifest shape:",
    configuration_change_manifest.shape,
)

display(configuration_change_manifest.head())

Global manifest shape: (22287, 28)


,record_id,repository,repository_url,dataset_split,repository_commit_index,commit_hash,parent_hash,commit_author_date,commit_committer_date,commit_message,...,change_type,configuration_type,added_lines,deleted_lines,old_size_bytes,new_size_bytes,old_line_count,new_line_count,source_before_sha256,source_after_sha256
0,f3e711748b8342bed50d50afa170591f3dd0cdd2df05b0...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,0,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,...,MODIFY,helm,2,2,1747,1747,38,38,0cdfd4aeb97450ca85844360b155948a43e2266793ea25...,b8b3460ce2131db8f4e306ed7d2fd337036390ad1f733f...
1,4ada560261335ebf7d43ecb68fcd56a2f0f3d380286a6b...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,0,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,...,MODIFY,helm,1,1,5282,5286,220,220,6fe21455e7f264bf9152fb21ee8454e5d16165c33f924b...,4cc974ac64ec7fb513a6081e267d91448bd76a236e0b4c...
2,3cb9882df2d988155d0ea9c634473aa2bb22ba373bad1b...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,0,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,...,MODIFY,kustomize,1,1,2102,2106,88,88,53b76b55282942e74e27ff40e168209921e2dcddbbf7b8...,01f805ec9d094304a6a22d6f3f0176a222f242d1592c65...
3,c213b0bf8781befc5354bd284fa14c4cee54b360bb75a5...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,0,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,...,MODIFY,kustomize,1,1,3460,3464,156,156,9e628a60c1319fe0dffc7ec94faa3f61ff694cf13d28ab...,e4c19590f89bc737b280739f04d531f42d3f0e9a82e36e...
4,bdd2333aece93ac5545c7a1445480ecd1e188719c8f181...,microservices_demo,https://github.com/GoogleCloudPlatform/microse...,train,0,9a4616e77f0f9cbcbecaf27d711c38890dda1404,ea0fcf429ff59d861867370d75586095dd253813,2026-07-13T10:40:02-04:00,2026-07-13T10:40:02-04:00,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,...,MODIFY,kustomize,1,1,2543,2547,95,95,14a4cfc52711df1eec6636c35e0264ff775228d5042646...,4e3872693c0ee772fe7617d2f9696246aaefcdaba37276...


In [13]:
manifest_csv_path = (
    RAW_DATA_DIR
    / "configuration_change_manifest.csv"
)

manifest_json_path = (
    RAW_DATA_DIR
    / "configuration_change_manifest.json"
)

summary_csv_path = (
    RAW_DATA_DIR
    / "repository_extraction_summary.csv"
)

configuration_change_manifest.to_csv(
    manifest_csv_path,
    index=False,
)

with manifest_json_path.open("w", encoding="utf-8") as file:
    json.dump(
        configuration_change_manifest.to_dict(
            orient="records"
        ),
        file,
        indent=2,
        default=str,
    )

extraction_summary_df.to_csv(
    summary_csv_path,
    index=False,
)

print("Saved global metadata manifest:")
print("CSV :", manifest_csv_path)
print("JSON:", manifest_json_path)
print("\nSaved repository summary:")
print(summary_csv_path)

Saved global metadata manifest:
CSV : C:\Users\Lenovo\Desktop\DriftGuard\data\raw\configuration_change_manifest.csv
JSON: C:\Users\Lenovo\Desktop\DriftGuard\data\raw\configuration_change_manifest.json

Saved repository summary:
C:\Users\Lenovo\Desktop\DriftGuard\data\raw\repository_extraction_summary.csv


In [14]:
print("Configuration changes by dataset split:")

split_distribution = (
    configuration_change_manifest[
        "dataset_split"
    ]
    .value_counts()
    .rename_axis("dataset_split")
    .reset_index(name="change_records")
)

display(split_distribution)

print("\nConfiguration changes by repository:")

repository_distribution = (
    configuration_change_manifest
    .groupby(
        ["dataset_split", "repository"],
        as_index=False,
    )
    .agg(
        change_records=("record_id", "count"),
        unique_commits=("commit_hash", "nunique"),
        unique_files=("effective_path", "nunique"),
    )
    .sort_values(
        ["dataset_split", "change_records"],
        ascending=[True, False],
    )
)

display(repository_distribution)

print("\nConfiguration changes by technology:")

configuration_type_distribution = (
    configuration_change_manifest
    .groupby(
        ["dataset_split", "configuration_type"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "change_records"})
    .sort_values(
        ["dataset_split", "change_records"],
        ascending=[True, False],
    )
)

display(configuration_type_distribution)

Configuration changes by dataset split:


,dataset_split,change_records
0,train,11709
1,test,8632
2,validation,1946



Configuration changes by repository:


,dataset_split,repository,change_records,unique_commits,unique_files
1,test,terraform_eks_blueprints,7700,1142,1731
0,test,ansible_examples,932,230,331
2,train,kube_prometheus,7253,1297,455
3,train,microservices_demo,3477,1205,181
4,train,terraform_aws_vpc,979,287,110
5,validation,kubernetes_examples,1946,624,669



Configuration changes by technology:


,dataset_split,configuration_type,change_records
7,test,terraform,6178
9,test,yaml,1240
0,test,ansible,588
3,test,helm,244
5,test,kubernetes,227
6,test,nginx,105
4,test,json,24
1,test,configuration,20
2,test,docker,5
8,test,toml,1


In [15]:
print("Git change types:")

change_type_distribution = (
    configuration_change_manifest[
        "change_type"
    ]
    .value_counts(dropna=False)
    .rename_axis("change_type")
    .reset_index(name="count")
)

display(change_type_distribution)

configuration_change_manifest[
    "commit_author_date"
] = pd.to_datetime(
    configuration_change_manifest[
        "commit_author_date"
    ],
    errors="coerce",
    utc=True,
)

temporal_coverage = (
    configuration_change_manifest
    .groupby(
        ["dataset_split", "repository"],
        as_index=False,
    )
    .agg(
        earliest_change=(
            "commit_author_date",
            "min",
        ),
        latest_change=(
            "commit_author_date",
            "max",
        ),
        unique_commits=(
            "commit_hash",
            "nunique",
        ),
    )
)

display(temporal_coverage)

Git change types:


,change_type,count
0,MODIFY,17702
1,ADD,2679
2,DELETE,1653
3,RENAME,253


,dataset_split,repository,earliest_change,latest_change,unique_commits
0,test,ansible_examples,2013-03-12 07:35:13+00:00,2020-07-19 23:39:16+00:00,230
1,test,terraform_eks_blueprints,2021-04-07 00:43:01+00:00,2026-07-07 16:04:37+00:00,1142
2,train,kube_prometheus,2016-10-18 09:29:18+00:00,2026-07-17 22:52:36+00:00,1297
3,train,microservices_demo,2018-06-14 18:03:02+00:00,2026-07-13 14:40:02+00:00,1205
4,train,terraform_aws_vpc,2017-08-28 18:50:19+00:00,2026-04-02 20:21:35+00:00,287
5,validation,kubernetes_examples,2014-06-06 23:40:48+00:00,2026-02-28 09:59:04+00:00,624


In [16]:
def truncate_text(value, maximum_length=300):
    if value is None:
        return None

    value = str(value).replace("\r", "")

    if len(value) <= maximum_length:
        return value

    return value[:maximum_length] + " ..."


sample_records = []

for repository in REPOSITORIES:
    repository_name = repository["name"]

    data_path = (
        RAW_DATA_DIR
        / f"{repository_name}_configuration_changes.jsonl.gz"
    )

    if not data_path.exists():
        continue

    collected_for_repository = 0

    with gzip.open(
        data_path,
        mode="rt",
        encoding="utf-8",
    ) as input_file:

        for line in input_file:
            record = json.loads(line)

            sample_records.append(
                {
                    "repository": record["repository"],
                    "split": record["dataset_split"],
                    "commit_hash":
                        record["commit_hash"][:10],
                    "configuration_type":
                        record["configuration_type"],
                    "change_type":
                        record["change_type"],
                    "file":
                        record["effective_path"],
                    "commit_message":
                        truncate_text(
                            record["commit_message"],
                            120,
                        ),
                    "source_before":
                        truncate_text(
                            record["source_before"],
                            250,
                        ),
                    "source_after":
                        truncate_text(
                            record["source_after"],
                            250,
                        ),
                }
            )

            collected_for_repository += 1

            if collected_for_repository >= 2:
                break

sample_df = pd.DataFrame(sample_records)

display(sample_df)

,repository,split,commit_hash,configuration_type,change_type,file,commit_message,source_before,source_after
0,microservices_demo,train,9a4616e77f,helm,MODIFY,helm-chart/Chart.yaml,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,# Copyright 2023 Google LLC\n#\n# Licensed und...,# Copyright 2023 Google LLC\n#\n# Licensed und...
1,microservices_demo,train,9a4616e77f,helm,MODIFY,helm-chart/values.yaml,release/v0.10.6 (#3432)\n\n* Release v0.10.6\n...,# Copyright 2024 Google LLC\n#\n# Licensed und...,# Copyright 2024 Google LLC\n#\n# Licensed und...
2,kube_prometheus,train,fe1b0c1b85,yaml,MODIFY,.github/workflows/ci.yaml,Bump actions/setup-go from 6 to 7\n\nBumps [ac...,name: ci\non:\n - push\n - pull_request\njob...,name: ci\non:\n - push\n - pull_request\njob...
3,kube_prometheus,train,6c6571ce43,yaml,MODIFY,.github/workflows/ci.yaml,bump actions/setup-go from 6 to 7\n\nBumps [ac...,name: ci\non:\n - push\n - pull_request\njob...,name: ci\non:\n - push\n - pull_request\njob...
4,terraform_aws_vpc,train,7a28ce8ec6,terraform,MODIFY,modules/vpc-endpoints/main.tf,fix: Correct `dns_options.dns_record_ip_type` ...,##############################################...,##############################################...
5,terraform_aws_vpc,train,1c561a729a,yaml,MODIFY,.pre-commit-config.yaml,feat: Add provider meta user-agent (#1272)\n\n...,repos:\n - repo: https://github.com/antonbabe...,repos:\n - repo: https://github.com/antonbabe...
6,kubernetes_examples,validation,7d0da27cd0,yaml,MODIFY,web/guestbook/frontend-deployment.yaml,Improve guestbook frontend deployment by addin...,apiVersion: apps/v1 # for k8s versions before...,apiVersion: apps/v1 # for k8s versions before...
7,kubernetes_examples,validation,6227892d66,nginx,ADD,nginx-platform-app/deployment.yml,add nginx platform application example with pr...,NaN,apiVersion: apps/v1\nkind: Deployment\nmetadat...
8,ansible_examples,test,3356f4876f,yaml,MODIFY,.github/workflows/main.yml,YAML doesn't need quotes\n\nCo-Authored-By: Sv...,"name: Ansible Lint\n\non: [push, pull_request]...","name: Ansible Lint\n\non: [push, pull_request]..."
9,ansible_examples,test,d8885d15ef,yaml,MODIFY,.github/workflows/main.yml,Use recommended workflow\n\nhttps://github.com...,name: Lint\n\non:\n push:\n branches: [ ma...,"name: Ansible Lint\n\non: [push, pull_request]..."


In [17]:
repositories_by_split = (
    configuration_change_manifest
    .groupby("dataset_split")["repository"]
    .apply(set)
    .to_dict()
)

train_repositories = repositories_by_split.get(
    "train",
    set(),
)

validation_repositories = repositories_by_split.get(
    "validation",
    set(),
)

test_repositories = repositories_by_split.get(
    "test",
    set(),
)

repository_overlap_checks = {
    "Train and validation repositories are disjoint":
        train_repositories.isdisjoint(
            validation_repositories
        ),

    "Train and test repositories are disjoint":
        train_repositories.isdisjoint(
            test_repositories
        ),

    "Validation and test repositories are disjoint":
        validation_repositories.isdisjoint(
            test_repositories
        ),
}

for check_name, passed in repository_overlap_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status:<8} | {check_name}")

assert all(repository_overlap_checks.values())

print("\nTrain repositories     :", sorted(train_repositories))
print("Validation repositories:", sorted(validation_repositories))
print("Test repositories      :", sorted(test_repositories))

PASSED   | Train and validation repositories are disjoint
PASSED   | Train and test repositories are disjoint
PASSED   | Validation and test repositories are disjoint

Train repositories     : ['kube_prometheus', 'microservices_demo', 'terraform_aws_vpc']
Validation repositories: ['kubernetes_examples']
Test repositories      : ['ansible_examples', 'terraform_eks_blueprints']


In [18]:
integrity_checks = {
    "Manifest is not empty":
        len(configuration_change_manifest) > 0,

    "Every record has a unique record ID":
        configuration_change_manifest[
            "record_id"
        ].is_unique,

    "Every record has a repository":
        configuration_change_manifest[
            "repository"
        ].notna().all(),

    "Every record has a dataset split":
        configuration_change_manifest[
            "dataset_split"
        ].isin(
            ["train", "validation", "test"]
        ).all(),

    "Every record has a commit hash":
        configuration_change_manifest[
            "commit_hash"
        ].notna().all(),

    "Every record has an effective path":
        configuration_change_manifest[
            "effective_path"
        ].notna().all(),

    "No merge commits were retained":
        (
            configuration_change_manifest[
                "is_merge_commit"
            ] == False
        ).all(),

    "No extracted file exceeds the size limit":
        (
            configuration_change_manifest[
                [
                    "old_size_bytes",
                    "new_size_bytes",
                ]
            ].max(axis=1)
            <= MAX_FILE_SIZE_BYTES
        ).all(),

    "Training data exists":
        (
            configuration_change_manifest[
                "dataset_split"
            ] == "train"
        ).any(),

    "Validation data exists":
        (
            configuration_change_manifest[
                "dataset_split"
            ] == "validation"
        ).any(),

    "Unseen test data exists":
        (
            configuration_change_manifest[
                "dataset_split"
            ] == "test"
        ).any(),
}

print("Data-integrity checks:\n")

for check_name, passed in integrity_checks.items():
    status = "PASSED" if passed else "FAILED"
    print(f"{status:<8} | {check_name}")

all_integrity_checks_passed = all(
    integrity_checks.values()
)

print("\n" + "=" * 72)

if all_integrity_checks_passed:
    print("ALL DATA-INTEGRITY CHECKS PASSED")
else:
    print("ONE OR MORE DATA-INTEGRITY CHECKS FAILED")

print("=" * 72)

Data-integrity checks:

PASSED   | Manifest is not empty
PASSED   | Every record has a unique record ID
PASSED   | Every record has a repository
PASSED   | Every record has a dataset split
PASSED   | Every record has a commit hash
PASSED   | Every record has an effective path
PASSED   | No merge commits were retained
PASSED   | No extracted file exceeds the size limit
PASSED   | Training data exists
PASSED   | Validation data exists
PASSED   | Unseen test data exists

ALL DATA-INTEGRITY CHECKS PASSED


In [19]:
skip_summary_records = []

for summary in extraction_summaries:
    repository_name = summary["repository"]

    for reason, count in summary[
        "skip_reasons"
    ].items():
        skip_summary_records.append(
            {
                "repository": repository_name,
                "split": summary["dataset_split"],
                "reason": reason,
                "count": count,
            }
        )

skip_summary_df = pd.DataFrame(skip_summary_records)

if skip_summary_df.empty:
    print("No skipped-file records were reported.")
else:
    display(
        skip_summary_df.sort_values(
            "count",
            ascending=False,
        )
    )

total_extraction_errors = sum(
    summary["extraction_errors"]
    for summary in extraction_summaries
)

print("\nTotal extraction errors:", total_extraction_errors)

,repository,split,reason,count
0,microservices_demo,train,not_configuration_file,4616
7,kubernetes_examples,validation,not_configuration_file,3462
2,kube_prometheus,train,not_configuration_file,3314
11,terraform_eks_blueprints,test,not_configuration_file,3275
5,terraform_aws_vpc,train,not_configuration_file,753
10,ansible_examples,test,not_configuration_file,751
12,terraform_eks_blueprints,test,missing_or_binary_source,725
8,kubernetes_examples,validation,missing_or_binary_source,447
4,kube_prometheus,train,missing_or_binary_source,227
3,kube_prometheus,train,file_too_large,187



Total extraction errors: 0


In [20]:
total_records = len(configuration_change_manifest)

total_unique_commits = (
    configuration_change_manifest[
        "commit_hash"
    ].nunique()
)

total_unique_files = (
    configuration_change_manifest[
        [
            "repository",
            "effective_path",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

total_repositories = (
    configuration_change_manifest[
        "repository"
    ].nunique()
)

summary_by_split = (
    configuration_change_manifest
    .groupby("dataset_split")
    .agg(
        records=("record_id", "count"),
        repositories=("repository", "nunique"),
        commits=("commit_hash", "nunique"),
        files=("effective_path", "nunique"),
    )
)

print("=" * 72)
print("NOTEBOOK 02 COMPLETED")
print("=" * 72)

print(f"Repositories processed       : {total_repositories:,}")
print(f"Configuration change records : {total_records:,}")
print(f"Unique configuration commits : {total_unique_commits:,}")
print(f"Unique repository-file pairs : {total_unique_files:,}")

print("\nDataset summary by split:")
display(summary_by_split)

print("\nImportant:")
print("- Test repositories remain untouched by model training.")
print("- Full before/after source contents are stored in compressed JSONL.")
print("- The CSV manifest contains metadata only.")
print("- Structural field-level diffs have not yet been generated.")

print("\nNext notebook:")
print("03_structural_diff_generation.ipynb")

NOTEBOOK 02 COMPLETED
Repositories processed       : 6
Configuration change records : 22,287
Unique configuration commits : 4,785
Unique repository-file pairs : 3,477

Dataset summary by split:


,records,repositories,commits,files
dataset_split,,,,
test,8632,2,1372,2062
train,11709,3,2789,745
validation,1946,1,624,669



Important:
- Test repositories remain untouched by model training.
- Full before/after source contents are stored in compressed JSONL.
- The CSV manifest contains metadata only.
- Structural field-level diffs have not yet been generated.

Next notebook:
03_structural_diff_generation.ipynb
